<a href="https://colab.research.google.com/github/andreaparrd/biodiversity-lulc/blob/main/scripts/1_data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Script 1 — `1_data_cleaning`: read raw data, clean it and write it to `clean_data/` subfolder

**Reminder:** read from `raw_data/`, write to `clean_data/`.

## Step 0. Mount Google Drive

Setting Google Colab's runtime to Python to use the drive.mount() function.
Reminder: change the run time to Python, then run the cell below.

In [1]:
# Runtime must be Python for this one block of code.
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Step 1. Set the working directory

In [1]:
# Point R at the project folder so every path below can be short
setwd("/content/drive/MyDrive/biodiversity_lulc_project")
getwd()

[1] "/content/drive/MyDrive/biodiversity_lulc_project"

In [2]:
#Load libraries

library(readr)  # read_csv(), write_csv()
library(dplyr)  # rename(), mutate(), filter(), left_join(), anti_join()
library(tidyr)  # pivot_longer()
library(lubridate)
library(stringr)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘lubridate’


The following objects are masked from ‘package:base’:

    date, intersect, setdiff, union




## Step 2. Read the raw data and look at it

In [3]:
# Read  raw files. This script only ever READS from raw_data/.
bio_monitoring  <- read_csv("raw_data/biodiversity_monitoring.csv")

Warning message:
“One or more parsing issues, call `problems()` on your data frame for details,
e.g.:
  dat <- vroom(...)
  problems(dat)”
Rows: 994 Columns: 1
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (1): FID *;tipo_registro;Clase;Estado_Nacional;Estado_UICN;estado_CITES;...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [4]:
head (bio_monitoring)

FID *;tipo_registro;Clase;Estado_Nacional;Estado_UICN;estado_CITES;Fecha;Nombre_cientifico;Nombre_comun;Nro_Individuos;Propiedad;X;Y;Tramo
<chr>
1540;Atropellado;Mammalia;<Nulo>;LC;Apéndice II;11/01/2023;Cerdocyon thous;Aguara’i;1;San Liberato;519613;7500825;<Nulo>
1541;Atropellado;Mammalia;<Nulo>;LC;No listada;11/01/2023;Euphractus sexcinctus;Tatú poyú;1;San Liberato;522364;7501840;<Nulo>
1542;Atropellado;Mammalia;<Nulo>;LC;Apéndice II;10/02/2023;Cerdocyon thous;Aguara’i;1;Trementina;516172;7485226;<Nulo>
1543;Atropellado;Mammalia;<Nulo>;LC;Apéndice II;15/03/2023;Cerdocyon thous;Aguara’i;1;Trementina;517374;7489126;<Nulo>
1544;Atropellado;Mammalia;<Nulo>;LC;No listada;16/03/2023;Euphractus sexcinctus;Tatú poyú;1;San Liberato;518423;7498975;<Nulo>
1545;Atropellado;Mammalia;<Nulo>;LC;Apéndice II;17/03/2023;Cerdocyon thous;Aguara’i;1;Trementina;516771;7485385;<Nulo>


In [5]:
glimpse (bio_monitoring)

Rows: 994
Columns: 1
$ `FID *;tipo_registro;Clase;Estado_Nacional;Estado_UICN;estado_CITES;Fecha;Nombre_cientifico;Nombre_comun;Nro_Individuos;Propiedad;X;Y;Tramo` <chr> …


# Findings
After inspecting the data we can see that due to the use of semicolons (;) instead of commas (,) as column separators, which is the standard default when exporting CSV files from Excel in Spanish, read_csv() can't read it properly because it only splits on commas.


## Step 3. Data cleaning and transformation

In [6]:
#Split the column into multiple new columns using a specific delimiter, semicolons (;) in this case

bio_monitoring_split <- bio_monitoring %>%
  separate_wider_delim(
    cols = 1,
    delim = ";",
    names = c(
      "FID", "tipo_registro", "Clase", "Estado_Nacional", "Estado_UICN",
      "estado_CITES", "Fecha", "Nombre_cientifico", "Nombre_comun",
      "Nro_Individuos", "Propiedad", "X", "Y", "Tramo"
    )
  )

glimpse(bio_monitoring_split)

Rows: 994
Columns: 14
$ FID               <chr> "1540", "1541", "1542", "1543", "1544", "1545", "154…
$ tipo_registro     <chr> "Atropellado", "Atropellado", "Atropellado", "Atrope…
$ Clase             <chr> "Mammalia", "Mammalia", "Mammalia", "Mammalia", "Mam…
$ Estado_Nacional   <chr> "<Nulo>", "<Nulo>", "<Nulo>", "<Nulo>", "<Nulo>", "<…
$ Estado_UICN       <chr> "LC", "LC", "LC", "LC", "LC", "LC", "LC", "<Nulo>", …
$ estado_CITES      <chr> "Apéndice II", "No listada", "Apéndice II", "Apéndic…
$ Fecha             <chr> "11/01/2023", "11/01/2023", "10/02/2023", "15/03/202…
$ Nombre_cientifico <chr> "Cerdocyon thous", "Euphractus sexcinctus", "Cerdocy…
$ Nombre_comun      <chr> "Aguara’i", "Tatú poyú", "Aguara’i", "Aguara’i", "Ta…
$ Nro_Individuos    <chr> "1", "1", "1", "1", "1", "1", "1", "1", "1", "1", "1…
$ Propiedad         <chr> "San Liberato", "San Liberato", "Trementina", "Treme…
$ X                 <chr> "519613", "522364", "516172", "517374", "518423", "5…
$ Y               

We were able to split the column into multiple columns, but we still need to convert the data types, standardize missing-value strings and convert comma decimals to periods.

In [8]:
bio_monitoring_clean <- bio_monitoring_split %>%
  # 1. Normalize and convert common text to true NA
  mutate(
    across(where(is.character), str_squish),
    across(where(is.character), ~ na_if(.x, "NA")),
    across(where(is.character), ~ na_if(.x, "N/A")),
    across(where(is.character), ~ na_if(.x, "S/D")),
    across(where(is.character), ~ na_if(.x, "<Nulo>"))
  ) %>%
  # 2. Converting columns to numeric and date types
  mutate(
    FID = as.numeric(FID),
    Fecha = dmy(Fecha),
    Nro_Individuos = as.numeric(Nro_Individuos),
    X = as.numeric(str_replace(X, ",", ".")),
    Y = as.numeric(str_replace(Y, ",", "."))
  )

# 3. Verify final results
glimpse(bio_monitoring_clean)



Rows: 994
Columns: 14
$ FID               <dbl> 1540, 1541, 1542, 1543, 1544, 1545, 1546, 1547, 1548…
$ tipo_registro     <chr> "Atropellado", "Atropellado", "Atropellado", "Atrope…
$ Clase             <chr> "Mammalia", "Mammalia", "Mammalia", "Mammalia", "Mam…
$ Estado_Nacional   <chr> NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, …
$ Estado_UICN       <chr> "LC", "LC", "LC", "LC", "LC", "LC", "LC", NA, "LC", …
$ estado_CITES      <chr> "Apéndice II", "No listada", "Apéndice II", "Apéndic…
$ Fecha             <date> 2023-01-11, 2023-01-11, 2023-02-10, 2023-03-15, 202…
$ Nombre_cientifico <chr> "Cerdocyon thous", "Euphractus sexcinctus", "Cerdocy…
$ Nombre_comun      <chr> "Aguara’i", "Tatú poyú", "Aguara’i", "Aguara’i", "Ta…
$ Nro_Individuos    <dbl> 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1…
$ Propiedad         <chr> "San Liberato", "San Liberato", "Trementina", "Treme…
$ X                 <dbl> 519613.0, 522364.0, 516172.0, 517374.0, 518423.0, 51…
$ Y               

# Step 4. Write it to clean_data subfolder

In [9]:
write_csv(bio_monitoring_clean, "clean_data/bio_monitoring_clean.csv")

# Check: the file should now be in clean_data/
list.files("clean_data")

[1] "bio_monitoring_clean.csv"